<a href="https://colab.research.google.com/github/rallyfranky/my-first-repo/blob/main/newspaper_budget_optimize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pulp
import pandas as pd
import sys
from __future__ import annotations

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/sites.csv')

,拠点番号,拠点都道府県,数値1,数値2
0,S00001,茨城県,714000,370
1,S00002,千葉県,485000,261
2,S00003,大阪府,1940000,969
3,S00004,山口県,771000,246
4,S00005,千葉県,446000,216


In [4]:
df = df.rename(columns={'拠点番号': 'branch_no',
                   '拠点都道府県':'pref',
                   '数値1': 'budget',
                   '数値2': 'kpi'})

In [32]:
def news_optimize(df: pd.DataFrame, budget: float,min_pref: float):
  df = df.reset_index(drop=True)
  idx = list(df.index)
  b = df['budget'].to_dict()
  k = df['kpi'].to_dict()
  prob = pulp.LpProblem('newspaper_buying', pulp.LpMaximize)
  #決定変数、x={0,1}拠点採択のバイナリ
  x = pulp.LpVariable.dicts('x', idx, cat=pulp.LpBinary)
  #目的変数（KPI最大化）
  prob += pulp.lpSum(x[i] * k[i] for i in idx), "total_kpi"
  #予算制約
  prob += pulp.lpSum(x[i] * b[i] for i in idx) <= budget, "total_budget"
  #地域制約
  for pref, member in df.groupby('pref').groups.items():
    prob += (
        pulp.lpSum(x[i] for i in member) >= 1,
        f"cover_{pref}",
    )
  prob.solve()

  print("Status:", pulp.LpStatus[prob.status])
  selected_branches = [i for i in idx if x[i].varValue == 1]
  total_kpi = pulp.value(prob.objective)
  total_budget_spent = sum(b[i] for i in selected_branches)
  selected_DF = pd.DataFrame(selected_branches)
  return selected_DF

In [34]:
optimized = news_optimize(df, 65000000, 1)

Status: Optimal


In [44]:
optmized = optimized.copy().rename(columns={0: 'idx'})

In [39]:
df['idx'] = df.index
df

,branch_no,pref,budget,kpi,idx
0,S00001,茨城県,714000,370,0
1,S00002,千葉県,485000,261,1
2,S00003,大阪府,1940000,969,2
3,S00004,山口県,771000,246,3
4,S00005,千葉県,446000,216,4
...,...,...,...,...,...
1195,S01196,広島県,1426000,398,1195
1196,S01197,新潟県,560000,184,1196
1197,S01198,愛知県,1751000,1141,1197
1198,S01199,埼玉県,1207000,419,1198


In [49]:
solved = pd.merge(optmized, df, left_on='idx', right_on='idx')

In [50]:
solved['budget'].sum()

np.int64(64995000)